# MMOE 训练与评估（AUC / GAUC / NDCG）

本 Notebook 使用 `models/mmoe.py` 的 `build_mmoe_model` 进行多任务训练，并在验证集输出：
- AUC（各任务 AUC + 主任务 AUC）
- GAUC（按 user_id 分组，基于主任务）
- NDCG@K（按 user_id 分组，基于主任务）

In [1]:
# ===== 1) 依赖导入与全局配置 =====
import os
import sys
from typing import Dict, List

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import ndcg_score, roc_auc_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from models.mmoe import build_mmoe_model

CONFIG = {
    "data_path": os.path.join(project_root, "data/ctr_data_500k.csv"),
    "task_candidates": ["click","follow", "like", "share"],
    "main_task": "click",
    "batch_size": 256,
    "num_epochs": 15,
    "learning_rate": 1e-3,
    "weight_decay": 1e-5,
    "dropout": 0.1,
    "expert_nums": 4,
    "expert_dnn_units": (128, 64),
    "gate_dnn_units": (128, 64),
    "task_tower_dnn_units": (64, 32),
    "val_ratio": 0.2,
    "random_state": 42,
    "num_workers": 0,
    "ndcg_k": 10,
}

torch.manual_seed(CONFIG["random_state"])
np.random.seed(CONFIG["random_state"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")
print(f"数据文件: {CONFIG['data_path']}")
print(f"主任务: {CONFIG['main_task']}")

使用设备: cuda
数据文件: d:\Machine_learning\Recommend\LHY\data/ctr_data_500k.csv
主任务: click


In [2]:
# ===== 2) 数据读取与 DataLoader =====
df = pd.read_csv(CONFIG["data_path"], na_values=["\\N", "NULL", "null", "None", ""])
print("数据形状:", df.shape)

TASKS = [t for t in CONFIG["task_candidates"] if t in df.columns]
if len(TASKS) == 0:
    raise ValueError("数据中未找到任务标签列，请检查 task_candidates 配置。")
if CONFIG["main_task"] not in TASKS:
    raise ValueError(f"main_task={CONFIG['main_task']} 不在可用任务中: {TASKS}")

exclude_cols = set(TASKS)
feature_cols = [c for c in df.columns if c not in exclude_cols]
if "user_id" not in feature_cols:
    raise ValueError("GAUC/NDCG 需要 user_id 列，请确保数据包含 user_id。")

X_raw = df[feature_cols].copy()
Y_raw = df[TASKS].copy()

# 强制将特征转为数值；无法解析的值（如 '\\N'）转为 NaN。
X = X_raw.apply(pd.to_numeric, errors="coerce")

# 任务标签也统一转为数值，兼容 True/False、0/1、字符串数字。
Y = Y_raw.apply(pd.to_numeric, errors="coerce").fillna(0.0).astype(np.float32)

# 保存原始 user_id（未标准化）用于分组指标。
user_ids = X["user_id"].copy()

# 用每列中位数填补缺失；整列缺失则填 0。
for col in feature_cols:
    if X[col].isna().all():
        X[col] = 0.0
    else:
        X[col] = X[col].fillna(X[col].median())

x_train, x_val, y_train, y_val, user_train, user_val = train_test_split(
    X,
    Y,
    user_ids,
    test_size=CONFIG["val_ratio"],
    random_state=CONFIG["random_state"],
    shuffle=True,
)

# 仅用训练集统计量做标准化，避免验证集信息泄漏。
train_mean = x_train.mean()
train_std = x_train.std().replace(0, 1.0)
x_train = (x_train - train_mean) / (train_std + 1e-8)
x_val = (x_val - train_mean) / (train_std + 1e-8)

# 供 evaluate() 使用，保证 GAUC/NDCG 按真实 user_id 分组。
VAL_USER_IDS = user_val.reset_index(drop=True).to_numpy()


class MMOEDataset(Dataset):
    def __init__(self, x_df: pd.DataFrame, y_df: pd.DataFrame):
        self.x = torch.tensor(x_df.reset_index(drop=True).values, dtype=torch.float32)
        self.y = torch.tensor(y_df.reset_index(drop=True).values, dtype=torch.float32)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx: int):
        return self.x[idx], self.y[idx]


train_dataset = MMOEDataset(x_train, y_train)
val_dataset = MMOEDataset(x_val, y_val)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=torch.cuda.is_available(),
)

print("任务:", TASKS)
print("特征维度:", len(feature_cols))
print("训练样本:", len(train_dataset), "验证样本:", len(val_dataset))

数据形状: (500000, 20)
任务: ['click', 'follow', 'like', 'share']
特征维度: 16
训练样本: 400000 验证样本: 100000


In [3]:
# ===== 3) 指标函数（AUC / GAUC / NDCG） =====
def safe_auc(y_true: np.ndarray, y_score: np.ndarray) -> float:
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_score))


def gauc_score(y_true: np.ndarray, y_score: np.ndarray, group_ids: np.ndarray) -> float:
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    group_ids = np.asarray(group_ids)

    weighted_auc_sum = 0.0
    weight_sum = 0

    for uid in np.unique(group_ids):
        mask = group_ids == uid
        y_u = y_true[mask]
        p_u = y_score[mask]
        if len(y_u) < 2 or len(np.unique(y_u)) < 2:
            continue
        auc_u = roc_auc_score(y_u, p_u)
        w = len(y_u)
        weighted_auc_sum += auc_u * w
        weight_sum += w

    if weight_sum == 0:
        return float("nan")
    return float(weighted_auc_sum / weight_sum)


def ndcg_at_k(y_true: np.ndarray, y_score: np.ndarray, group_ids: np.ndarray, k: int = 10) -> float:
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    group_ids = np.asarray(group_ids)

    ndcgs = []
    for uid in np.unique(group_ids):
        mask = group_ids == uid
        y_u = y_true[mask]
        p_u = y_score[mask]
        if len(y_u) < 2:
            continue
        # sklearn.ndcg_score 输入 shape [1, n_items]
        ndcgs.append(ndcg_score([y_u], [p_u], k=min(k, len(y_u))))

    if len(ndcgs) == 0:
        return float("nan")
    return float(np.mean(ndcgs))

In [4]:
# ===== 4) 构建模型与训练 =====
model = build_mmoe_model(
    feature_columns=len(feature_cols),
    task_name_list=TASKS,
    expert_nums=CONFIG["expert_nums"],
    expert_dnn_units=CONFIG["expert_dnn_units"],
    gate_dnn_units=CONFIG["gate_dnn_units"],
    task_tower_dnn_units=CONFIG["task_tower_dnn_units"],
    dropout=CONFIG["dropout"],
).to(device)

optimizer = optim.Adam(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
)
criterion = nn.BCEWithLogitsLoss()


def train_one_epoch() -> float:
    model.train()
    total_loss = 0.0
    n_batches = 0

    for xb, yb in tqdm(train_loader, desc="Train", leave=False):
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        logits_list, _ = model(xb, return_prob=False)
        loss = 0.0
        for i in range(len(TASKS)):
            task_logit = logits_list[i].squeeze(-1)
            task_label = yb[:, i]
            loss = loss + criterion(task_logit, task_label)
        loss = loss / len(TASKS)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / max(n_batches, 1)


def evaluate() -> Dict[str, float]:
    model.eval()

    probs_by_task: Dict[str, List[float]] = {t: [] for t in TASKS}
    labels_by_task: Dict[str, List[float]] = {t: [] for t in TASKS}
    user_ids_all: List[float] = []
    offset = 0

    with torch.no_grad():
        for xb, yb in tqdm(val_loader, desc="Eval", leave=False):
            bs = xb.size(0)
            user_ids_all.extend(VAL_USER_IDS[offset : offset + bs].tolist())
            offset += bs

            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            logits_list, _ = model(xb, return_prob=False)
            for i, t in enumerate(TASKS):
                prob = torch.sigmoid(logits_list[i].squeeze(-1))
                probs_by_task[t].extend(prob.cpu().numpy().tolist())
                labels_by_task[t].extend(yb[:, i].cpu().numpy().tolist())

    metrics = {}
    for t in TASKS:
        metrics[f"auc_{t}"] = safe_auc(np.array(labels_by_task[t]), np.array(probs_by_task[t]))

    main_t = CONFIG["main_task"]
    y_true_main = np.array(labels_by_task[main_t])
    y_prob_main = np.array(probs_by_task[main_t])
    group_ids = np.array(user_ids_all)

    metrics["auc"] = safe_auc(y_true_main, y_prob_main)
    metrics["gauc"] = gauc_score(y_true_main, y_prob_main, group_ids)
    metrics["ndcg"] = ndcg_at_k(y_true_main, y_prob_main, group_ids, k=CONFIG["ndcg_k"])
    metrics["ndgc"] = metrics["ndcg"]  # 兼容常见拼写
    return metrics

In [5]:
# ===== 5) 启动训练并输出验证指标 =====
best_auc = -1.0
best_metrics = None

for epoch in range(CONFIG["num_epochs"]):
    train_loss = train_one_epoch()
    metrics = evaluate()

    if not np.isnan(metrics["auc"]) and metrics["auc"] > best_auc:
        best_auc = metrics["auc"]
        best_metrics = metrics.copy()

    print(
        f"Epoch {epoch + 1}/{CONFIG['num_epochs']} | "
        f"loss={train_loss:.4f} | "
        f"AUC={metrics['auc']:.4f} | "
        f"GAUC={metrics['gauc']:.4f} | "
        f"NDCG@{CONFIG['ndcg_k']}={metrics['ndcg']:.4f}"
    )

print("\n训练完成")
if best_metrics is not None:
    print("最佳验证指标:")
    print(
        f"AUC={best_metrics['auc']:.4f}, "
        f"GAUC={best_metrics['gauc']:.4f}, "
        f"NDCG@{CONFIG['ndcg_k']}={best_metrics['ndcg']:.4f}"
    )

print("各任务 AUC(最后一轮):")
for t in TASKS:
    v = metrics.get(f"auc_{t}", float("nan"))
    print(f"  {t}: {v:.4f}")

Epoch 1/15 | loss=0.1392 | AUC=0.8049 | GAUC=0.7796 | NDCG@10=0.6556


Epoch 2/15 | loss=0.1278 | AUC=0.8094 | GAUC=0.7808 | NDCG@10=0.6565


Epoch 3/15 | loss=0.1269 | AUC=0.8115 | GAUC=0.7792 | NDCG@10=0.6562


Epoch 4/15 | loss=0.1260 | AUC=0.8149 | GAUC=0.7820 | NDCG@10=0.6575


Epoch 5/15 | loss=0.1254 | AUC=0.8167 | GAUC=0.7807 | NDCG@10=0.6562


...

Epoch 11/15 | loss=0.1224 | AUC=0.8259 | GAUC=0.7820 | NDCG@10=0.6578


Epoch 12/15 | loss=0.1221 | AUC=0.8277 | GAUC=0.7799 | NDCG@10=0.6567


Epoch 13/15 | loss=0.1218 | AUC=0.8273 | GAUC=0.7800 | NDCG@10=0.6568


Epoch 14/15 | loss=0.1214 | AUC=0.8295 | GAUC=0.7808 | NDCG@10=0.6579


Epoch 15/15 | loss=0.1213 | AUC=0.8302 | GAUC=0.7802 | NDCG@10=0.6558

训练完成
最佳验证指标:
AUC=0.8302, GAUC=0.7802, NDCG@10=0.6558
各任务 AUC(最后一轮):
  click: 0.8302
  follow: 0.7335
  like: 0.8698
  share: 0.8422
